In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Cross-Correlation

Analyze the time-lagged relationship between two signals using cross-correlation.

```{admonition} Goal
:class: tip
Analyze the time-lagged correlation between outdoor temperature and indoor temperature (Flat A). Cross-correlation reveals how strongly and with what delay the outdoor climate influences the indoor environment.
```

```{admonition} Data Basis
:class: note
**Datasets:**  
- `flatTempHum.csv` — Hourly indoor temperature (`FlatA_Temp`)  
- `centralOutsideTemp.csv` — Hourly outdoor temperature (`centralOutsideTemp`)  

**Time range:** Overlapping period of both datasets  
**Lag range:** -48 to +48 hours
```

## Data Preparation

In [ ]:
import pandas as pd

base_url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/"

# Load indoor temperature
df_indoor = pd.read_csv(base_url + "flatTempHum.csv", sep=";", usecols=["time", "FlatA_Temp"])
df_indoor["time"] = pd.to_datetime(df_indoor["time"])
df_indoor = df_indoor.set_index("time", drop=True)

# Load outdoor temperature
df_outdoor = pd.read_csv(base_url + "centralOutsideTemp.csv", sep=";")
df_outdoor["time"] = pd.to_datetime(df_outdoor["time"])
df_outdoor = df_outdoor.set_index("time", drop=True)

df_indoor.head()

In [ ]:
# Merge on time index and drop missing values
df = df_indoor.join(df_outdoor, how="inner").dropna()
df = df.rename(columns={"centralOutsideTemp": "outdoor", "FlatA_Temp": "indoor"})
print(f"Combined dataset: {len(df)} rows")
df.head()

## Visualization

In [ ]:
import numpy as np
import plotly.express as px

# Compute cross-correlation for lags -48 to +48 hours
max_lag = 48
lags = range(-max_lag, max_lag + 1)
correlations = []

for lag in lags:
    corr = df["outdoor"].corr(df["indoor"].shift(lag))
    correlations.append(corr)

df_corr = pd.DataFrame({"lag_hours": list(lags), "correlation": correlations})

# Find the lag with maximum correlation
best = df_corr.loc[df_corr["correlation"].idxmax()]
print(f"Maximum correlation: {best['correlation']:.3f} at lag = {int(best['lag_hours'])} hours")

In [ ]:
fig = px.bar(
    df_corr,
    x="lag_hours",
    y="correlation",
    title="Cross-Correlation: Outdoor vs. Indoor Temperature (Flat A)",
    labels={"lag_hours": "Lag [hours]", "correlation": "Pearson Correlation"},
)
fig.update_layout(bargap=0)
fig.show()

::::{dropdown} Customization
- Adjust `max_lag` to explore longer or shorter delay windows.
- A positive lag means indoor temperature lags behind outdoor temperature (thermal inertia of the building).
- Use `np.correlate` for a faster computation on large datasets (requires normalization).
- Replace `px.bar` with `px.line` for a smoother correlation curve.
::::